# AI Application Example III: Multi-Class Gas Classification
## Gas Sensor Array Drift at Different Concentrations

**V1 objective:** parse the UCI `.dat` files, validate the dataset, perform exploratory analysis, and establish a leakage-aware modelling foundation.

**Classification target:** six gases — Ethanol, Ethylene, Ammonia, Acetaldehyde, Acetone, and Toluene.

**Primary predictors:** 128 sensor-derived features.

**Metadata retained for analysis:** batch and concentration. They are not primary predictors.


In [3]:
# 1. Imports and reproducibility
import re
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Extracting dataset

The notebook expects the uploaded `.dat`. It searches common paths so it can also be moved to Colab.


In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("C:\\Users\\srian\\Documents\\GitHub\\42172 Introduction to Artificial Intelligence\\Data\\gas sensor array drift dataset at different concentrations")

gas_mapping = {
    1: "Ethanol",
    2: "Ethylene",
    3: "Ammonia",
    4: "Acetaldehyde",
    5: "Acetone",
    6: "Toluene"
}

all_data = []

for batch_file in sorted(DATA_DIR.glob("batch*.dat")):

    batch_id = int(batch_file.stem.replace("batch", ""))

    with open(batch_file, "r") as f:
        for line in f:

            parts = line.strip().split()

            # First token: class;concentration
            class_id, concentration = parts[0].split(";")

            row = {
                "batch": batch_id,
                "class_id": int(class_id),
                "gas": gas_mapping[int(class_id)],
                "concentration_ppmv": float(concentration)
            }

            # Remaining tokens: feature:value
            for item in parts[1:]:
                feature, value = item.split(":")
                row[f"feature_{feature}"] = float(value)

            all_data.append(row)

df = pd.DataFrame(all_data)

print(df.shape)
df.head()

(13910, 132)


,batch,class_id,gas,concentration_ppmv,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,...,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128
0,1,1,Ethanol,10.0,15596.1621,1.868245,2.371604,2.803678,7.512213,-2.739388,...,-1.071137,-3.037772,3037.0390,3.972203,0.527291,0.728443,1.445783,-0.545079,-0.902241,-2.654529
1,1,1,Ethanol,20.0,26402.0704,2.532401,5.411209,6.509906,7.658469,-4.722217,...,-1.530519,-1.994993,4176.4453,4.281373,0.980205,1.628050,1.951172,-0.889333,-1.323505,-1.749225
2,1,1,Ethanol,30.0,42103.5820,3.454189,8.198175,10.508439,11.611003,-7.668313,...,-2.384784,-2.867291,5914.6685,5.396827,1.403973,2.476956,3.039841,-1.334558,-1.993659,-2.348370
3,1,1,Ethanol,40.0,42825.9883,3.451192,12.113940,16.266853,39.910056,-7.849409,...,-2.607199,-3.058086,6147.4744,5.501071,1.981933,3.569823,4.049197,-1.432205,-2.146158,-2.488957
4,1,1,Ethanol,50.0,58151.1757,4.194839,11.455096,15.715298,17.654915,-11.083364,...,-3.594763,-4.181920,8158.6449,7.174334,1.993808,3.829303,4.402448,-1.930107,-2.931265,-4.088756
